# Simulating Human Sleep-Stage Transitions and Nighttime Sleep Fragmentation

This notebook develops a four-state, 30-second sleep-stage model from manually scored Sleep-EDF Expanded hypnograms. This milestone downloads a fixed pilot subset, prepares reproducible training and held-out data, and checks the transition counts.

## 1. Problem and question

How does increasing the probability of nighttime awakenings affect expected Deep Sleep and REM duration? The planned study will compare an empirical baseline with hypothetical changes in sleep-to-Wake transition probabilities, measuring stage minutes and fragmentation across replicated nights.

## 2. Model, assumptions and equations

Let $X_t$ be the stage in epoch $t$, where each epoch is 30 seconds and the possible states are Wake, Light, Deep and REM. The first-order, time-homogeneous DTMC assumes

$$P_{ij}=\Pr(X_{t+1}=j\mid X_t=i),\qquad \sum_j P_{ij}=1,\qquad \pi_{t+1}=\pi_t P.$$

Here $\pi_t$ is the row vector of state probabilities. Fit each dimensionless transition probability from training counts $C_{ij}$ as $\widehat P_{ij}=C_{ij}/\sum_k C_{ik}$ for rows with observed departures (including self-transitions). Estimate the initial distribution from the observed stages at the starts of training windows. A future simulator will sample from each row's cumulative probabilities using uniform random variates.

For a sleep-state row, if the baseline Wake probability is $q_i=P_{iW}<1$ and the scenario uses $q'_i$, set $P'_{iW}=q'_i$ and $P'_{ij}=P_{ij}(1-q'_i)/(1-q_i)$ for $j\ne W$. The Wake row stays fixed. A stage's total duration in minutes is its epoch count multiplied by $0.5$. An awakening is a sleep-to-Wake transition after sleep onset. The planned experiment uses independent night replications for means and 95% Monte Carlo confidence intervals.

The model pools training participants and assumes the same transition law throughout the analysis window. A homogeneous DTMC implies geometric stage run lengths. The grouped labels are derived from the historical Rechtschaffen–Kales scores. The scenarios are hypothetical model perturbations.

**Concepts applied** — DTMC state evolution, empirical input modelling and discrete inverse-transform sampling, and Monte Carlo estimation with the law of large numbers.

## 3. Implementation

In Colab, choose **Runtime → Run all**. The first code cell installs the EDF+ reader. The notebook downloads only the annotation files from the versioned PhysioNet source.

The fixed pilot manifest includes both listed nights for the first 12 Sleep Cassette participant IDs. A seeded participant split reserves three participants for later validation. This convenience subset keeps development repeatable and manageable. Its representativeness is unknown.

The provisional night window extends up to 30 minutes before the first and after the last scored non-Wake epoch, clipped to the annotated span. The padding follows the approach described in the MNE-Python sleep tutorial [3]. The same rule applies to every record. Original epoch indices remain relative to the recording. Movement, unscored intervals and uncovered epochs remain explicitly invalid. Only adjacent valid epochs within one recording contribute to transition counts.

In [ ]:
%pip install --quiet mne==1.10.2

In [ ]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
from urllib.error import URLError
from urllib.request import urlopen

import mne
import numpy as np
import pandas as pd
from IPython.display import display

SOURCE = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/"
MIRROR = "https://physionet-open.s3.amazonaws.com/sleep-edfx/1.0.0/sleep-cassette/"
RECORDS = """
SC4001EC SC4002EC SC4011EH SC4012EC SC4021EH SC4022EJ
SC4031EC SC4032EP SC4041EC SC4042EC SC4051EC SC4052EC
SC4061EC SC4062EC SC4071EC SC4072EH SC4081EC SC4082EP
SC4091EC SC4092EC SC4101EC SC4102EC SC4111EC SC4112EC
""".split()
STATES = ("Wake", "Light", "Deep", "REM")
STAGE_MAP = {
    "Sleep stage W": "Wake",
    "Sleep stage 1": "Light",
    "Sleep stage 2": "Light",
    "Sleep stage 3": "Deep",
    "Sleep stage 4": "Deep",
    "Sleep stage R": "REM",
    "Movement time": None,
    "Sleep stage ?": None,
}
EPOCH_SECONDS = 30
PADDING_EPOCHS = 30 * 60 // EPOCH_SECONDS
SPLIT_SEED = 4042

participants = sorted({record[:5] for record in RECORDS})
held_out = set(np.random.default_rng(SPLIT_SEED).choice(participants, size=3, replace=False))

In [ ]:
def expand_night(annotations):
    """Expand scored spans onto the recording's epoch grid and crop the window."""
    raw_labels = {}
    for onset, duration, label in zip(
        annotations.onset, annotations.duration, annotations.description, strict=True
    ):
        if label not in STAGE_MAP:
            raise ValueError(f"Unexpected Sleep-EDF annotation: {label!r}")
        start = round(onset / EPOCH_SECONDS)
        length = round(duration / EPOCH_SECONDS)
        if (
            start < 0
            or length <= 0
            or not np.isclose(onset, start * EPOCH_SECONDS, rtol=0, atol=1e-6)
            or not np.isclose(duration, length * EPOCH_SECONDS, rtol=0, atol=1e-6)
        ):
            raise ValueError(f"Invalid annotation timing: {onset=}, {duration=}, {label=}")
        for epoch in range(start, start + length):
            if epoch in raw_labels:
                raise ValueError(f"Overlapping annotations at epoch {epoch}")
            raw_labels[epoch] = label

    sleep = [
        epoch for epoch, label in raw_labels.items()
        if STAGE_MAP[label] in ("Light", "Deep", "REM")
    ]
    if not sleep:
        raise ValueError("No scored sleep in recording")
    first = max(min(raw_labels), min(sleep) - PADDING_EPOCHS)
    last = min(max(raw_labels), max(sleep) + PADDING_EPOCHS)
    night = pd.DataFrame({"epoch": range(first, last + 1)})
    night["label"] = night["epoch"].map(raw_labels).fillna("Uncovered")
    night["stage"] = night["label"].map(STAGE_MAP)
    return night


def count_transitions(night):
    """Count adjacent pairs with two valid stages in one recording."""
    following = night["stage"].shift(-1)
    adjacent = night["epoch"].shift(-1).sub(night["epoch"]).eq(1)
    counts = pd.crosstab(night.loc[adjacent, "stage"], following.loc[adjacent])
    return counts.reindex(index=STATES, columns=STATES, fill_value=0).astype("int64")


def fetch_annotation(filename):
    errors = []
    for base_url in (SOURCE, MIRROR):
        url = base_url + filename
        try:
            with urlopen(url, timeout=15) as response:
                return response.read(), response.geturl()
        except (URLError, TimeoutError) as exc:
            errors.append(f"{url}: {exc}")
    raise RuntimeError("Could not download annotation. " + " | ".join(errors))


nights, counts_by_record, audit_rows = {}, {}, []
with TemporaryDirectory(prefix="sleep-edf-") as temp:
    for record in RECORDS:
        participant = record[:5]
        split = "held-out" if participant in held_out else "training"
        filename = f"{record}-Hypnogram.edf"
        contents, url = fetch_annotation(filename)
        try:
            path = Path(temp) / filename
            path.write_bytes(contents)
            night = expand_night(mne.read_annotations(path))
        except (OSError, ValueError, RuntimeError) as exc:
            raise RuntimeError(f"Could not process {url}: {exc}") from exc

        counts = count_transitions(night)
        nights[record] = night
        counts_by_record[record] = counts
        audit_rows.append({
            "record": filename,
            "participant": participant,
            "split": split,
            "source_url": url,
            "sha256": sha256(contents).hexdigest(),
            "first_epoch": night["epoch"].iloc[0],
            "last_epoch": night["epoch"].iloc[-1],
            "window_epochs": len(night),
            "valid_epochs": night["stage"].notna().sum(),
            **{state: night["stage"].eq(state).sum() for state in STATES},
            "movement": night["label"].eq("Movement time").sum(),
            "unscored": night["label"].eq("Sleep stage ?").sum(),
            "uncovered": night["label"].eq("Uncovered").sum(),
            "valid_pairs": counts.to_numpy().sum(),
        })

audit = pd.DataFrame(audit_rows)
split_counts = {
    split: sum(counts_by_record[record] for record in RECORDS
               if (record[:5] in held_out) == (split == "held-out"))
    for split in ("training", "held-out")
}

## 4. Verification

Check that the split is disjoint, all window epochs are accounted for, and the count of valid pairs equals the number of valid epochs minus the number of contiguous valid runs, calculated separately for each recording.

In [ ]:
assert len(audit) == len(RECORDS) == len(set(RECORDS))
assert set(audit["split"]) == {"training", "held-out"}
assert audit.groupby("participant")["split"].nunique().eq(1).all()
assert audit.loc[audit["split"].eq("held-out"), "participant"].nunique() == len(held_out)
assert audit[list(STATES)].sum(axis=1).eq(audit["valid_epochs"]).all()
assert audit[["valid_epochs", "movement", "unscored", "uncovered"]].sum(axis=1).eq(
    audit["window_epochs"]
).all()
for record, night in nights.items():
    assert night["epoch"].diff().iloc[1:].eq(1).all()
    valid = night["stage"].notna()
    run_starts = valid & ~valid.shift(fill_value=False)
    assert counts_by_record[record].to_numpy().sum() == valid.sum() - run_starts.sum()
for split, counts in split_counts.items():
    assert counts.to_numpy().sum() == audit.loc[audit["split"].eq(split), "valid_pairs"].sum()
assert split_counts["training"].sum(axis=1).gt(0).all()
print("Preprocessing checks passed.")

## 5. Experiments and results

The tables audit the preprocessing inputs and transition counts. Rows in the count matrices are source stages and columns are destination stages. Held-out counts are kept separate from training. The planned scenario replications and confidence intervals follow in a later milestone.

In [ ]:
print(f"Source: {SOURCE}")
print(f"Participant split seed: {SPLIT_SEED}")
display(audit)
display(audit.groupby("split").agg(
    participants=("participant", "nunique"),
    recordings=("record", "size"),
    window_epochs=("window_epochs", "sum"),
    valid_epochs=("valid_epochs", "sum"),
    movement=("movement", "sum"),
    unscored=("unscored", "sum"),
    uncovered=("uncovered", "sum"),
    valid_pairs=("valid_pairs", "sum"),
))
for split, counts in split_counts.items():
    print(f"{split}: observed transition counts")
    display(counts)

## 6. Discussion and limitations

The first/last scored sleep rule is an approximation of the night window. Wake counts and estimated starting stages depend on the selected padding. Missing intervals interrupt transitions, so the matrices describe observed valid neighbours. The pilot is small, and pooled counts give more weight to recordings with more observed transitions. Later validation will compare simulated and held-out stage durations and awakening counts under consistent window and metric definitions.

## 7. References

1. B. Kemp, *Sleep-EDF Database Expanded*, v1.0.0, PhysioNet. [Dataset and annotation documentation](https://physionet.org/content/sleep-edfx/1.0.0/). DOI: 10.13026/C2X676. [ODC Attribution v1.0 licence](https://physionet.org/content/sleep-edfx/view-license/1.0.0/).
2. B. Kemp, A. H. Zwinderman, B. Tuk, H. A. C. Kamphuisen, and J. J. L. Oberyé, “Analysis of a sleep-dependent neuronal feedback loop: the slow-wave microcontinuity of the EEG,” *IEEE Transactions on Biomedical Engineering*, vol. 47, no. 9, pp. 1185–1194, 2000. [Publisher record](https://ieeexplore.ieee.org/document/867928/).
3. MNE-Python contributors, “Sleep stage classification from polysomnography (PSG) data.” [MNE-Python tutorial](https://mne.tools/stable/auto_tutorials/clinical/60_sleep.html#extract-30s-events-from-annotations).